# 🚀 CRAFT Phase 3: Evaluate, Compare & DeployThis notebook runs the final evaluation pipeline:| Step | What it does ||------|-------------|| **1a** | Clone repo + authenticate HF || **1b** | Install dependencies || **2** | Discover & mount all checkpoints (150, 200, 250) || **3** | Evaluate BASE model (Phi-3-Mini) || **4** | Evaluate all checkpoints (sweep) || **5** | Champion selection — find the best checkpoint || **6** | Merge + GGUF convert + quantize CHAMPION only || **7** | GGUF deployment verification || **8** | Generate benchmark report from REAL numbers || **9** | Upload to HuggingFace (gated on Samsung's bar) || **10** | Package delivery bundle |

---## Step 1: Setup Environment

In [ ]:
# 1a. Clone the repository + authenticate Hugging Face
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
git_token = user_secrets.get_secret("GITHUB_PAT")

# Authenticate Hugging Face globally for dataset downloads
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ Hugging Face Token authenticated globally!")
except Exception:
    print("⚠️ HF_TOKEN not found in secrets. Downloads might be rate-limited.")

repo_url = f"https://{git_token}@github.com/VishalGastu30/CRAFT-SLM-Reasoning.git"
if not os.path.exists("CRAFT-SLM-Reasoning"):
    !git clone {repo_url}
os.chdir("CRAFT-SLM-Reasoning")
!git pull
print("✅ Repository ready.")

In [ ]:
# Install all dependencies including llama-cpp-python for GGUF inference
!pip install -q transformers bitsandbytes accelerate datasets peft trl \
    loguru pyyaml scipy numpy huggingface_hub sentence-transformers

# Install llama-cpp-python with CUDA support for GGUF evaluation
# Comment this out and use the CPU version if CUDA install fails
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python \
    --upgrade --force-reinstall --no-cache-dir 2>/dev/null || \
    pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir

print("✅ All dependencies installed.")

---## Step 2: Multi-Checkpoint DiscoveryScans `/kaggle/input` for all available checkpoints (150, 200, 250) and copies them to the working directory.

In [ ]:
import os, shutil, json

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION — Set which checkpoints you uploaded to Kaggle input
# ═══════════════════════════════════════════════════════════════════
CHECKPOINTS_TO_EVALUATE = {
    "checkpoint-150": None,   # Will be filled by auto-discovery
    "checkpoint-200": None,
    "checkpoint-250": None,
}
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
# ═══════════════════════════════════════════════════════════════════

print("Scanning /kaggle/input for checkpoints...")
for root, dirs, files in os.walk('/kaggle/input'):
    for ck_name in CHECKPOINTS_TO_EVALUATE:
        if CHECKPOINTS_TO_EVALUATE[ck_name] is not None:
            continue  # Already found
        if ck_name in dirs:
            CHECKPOINTS_TO_EVALUATE[ck_name] = os.path.join(root, ck_name)
        elif 'adapter_model.safetensors' in files and ck_name.split('-')[-1] in root:
            CHECKPOINTS_TO_EVALUATE[ck_name] = root

# Copy found checkpoints to working directory
os.makedirs("checkpoints/rl", exist_ok=True)
for ck_name, ck_path in CHECKPOINTS_TO_EVALUATE.items():
    target = f"checkpoints/rl/{ck_name}"
    if ck_path and not os.path.exists(target):
        print(f"Copying {ck_name} from {ck_path}...")
        shutil.copytree(ck_path, target)
        print(f"  ✅ {ck_name} ready at {target}")
        print(f"  📁 Files: {os.listdir(target)}")
    elif os.path.exists(target):
        print(f"  ✅ {ck_name} already at {target}")
    else:
        print(f"  ⚠️  {ck_name} NOT FOUND in /kaggle/input — will be skipped")

# Summary
print("\n📋 Checkpoint discovery summary:")
for ck_name, ck_path in CHECKPOINTS_TO_EVALUATE.items():
    status = "✅ Found" if ck_path else "❌ Missing"
    print(f"  {ck_name}: {status}")

---## Step 3: Evaluate the Original Phi-3-Mini BaselineDownloads the untrained base model and runs it on GSM8K + StrategyQA to get the "before" score.

In [ ]:
import os
os.makedirs("craft_output", exist_ok=True)

# Evaluate the untrained base model — this is your "before" score
# Use 100 samples for reliable statistics (50 has too much variance)
!python -m src.phase3_deploy.evaluator \
    --model-type hf \
    --model-path microsoft/Phi-3-mini-4k-instruct \
    --dataset all \
    --samples 100 \
    --label "Phi-3-Mini-Baseline" \
    --gpu \
    --output-json craft_output/baseline_results.json

import json
with open("craft_output/baseline_results.json") as f:
    bl = json.load(f)
print("\n📊 Baseline Results:")
print(f"  GSM8K:      {bl['gsm8k']['summary']['accuracy']:.1%}")
print(f"  StrategyQA: {bl['strategyqa']['summary']['accuracy']:.1%}")
print("\n✅ Baseline evaluation complete.")

---## Step 4: Multi-Checkpoint Evaluation SweepEvaluates all available checkpoints (150, 200, 250) in HF format for a fair comparison against the baseline.

In [ ]:
import json

# Evaluate all available checkpoints in HF format
# This is the fair comparison — same format as baseline
checkpoint_results = {}

for ck_name in ["checkpoint-150", "checkpoint-200", "checkpoint-250"]:
    ck_path = f"checkpoints/rl/{ck_name}"
    results_path = f"craft_output/results_{ck_name.replace('-', '')}.json"
    
    if not os.path.exists(ck_path):
        print(f"⚠️  {ck_name} not found, skipping...")
        continue
    
    print(f"\n{'='*60}")
    print(f"Evaluating {ck_name}...")
    print(f"{'='*60}")
    
    !python -m src.phase3_deploy.evaluator \
        --model-type hf \
        --model-path {ck_path} \
        --dataset all \
        --samples 100 \
        --label "{ck_name}" \
        --gpu \
        --output-json {results_path}
    
    if os.path.exists(results_path):
        with open(results_path) as f:
            res = json.load(f)
        checkpoint_results[ck_name] = res
        gsm = res['gsm8k']['summary']['accuracy']
        strat = res['strategyqa']['summary']['accuracy']
        print(f"  ✅ {ck_name}: GSM8K={gsm:.1%}, StrategyQA={strat:.1%}")
    else:
        print(f"  ❌ Results file not created for {ck_name}. Check errors above.")

---## Step 5: 🏆 Champion Selection — The ArenaCompares all checkpoints, finds the one with the highest average improvement, and checks Samsung's requirements.

In [ ]:
import json

# ═══════════════════════════════════════════════════════════════════
# Samsung's required minimum scores
# ═══════════════════════════════════════════════════════════════════
SAMSUNG_TARGETS = {
    "gsm8k":      {"min_score": 0.50, "min_delta": 0.05},
    "strategyqa": {"min_score": 0.65, "min_delta": 0.05},
}
# ═══════════════════════════════════════════════════════════════════

with open("craft_output/baseline_results.json") as f:
    baseline = json.load(f)

baseline_gsm = baseline['gsm8k']['summary']['accuracy']
baseline_strat = baseline['strategyqa']['summary']['accuracy']

print(f"\n{'='*70}")
print(f"  CRAFT Checkpoint Sweep — The Arena")
print(f"{'='*70}")
print(f"\n  Samsung targets: GSM8K≥50% (+5%), StrategyQA≥65% (+5%)")
print(f"\n{'Checkpoint':<18} {'GSM8K':>8} {'Δ GSM':>8} {'StratQA':>9} {'Δ Strat':>9} {'Avg Δ':>7} {'Samsung':>9}")
print("-" * 70)

best_checkpoint = None
best_avg_delta = -999
all_results = {}

for ck_name in ["checkpoint-150", "checkpoint-200", "checkpoint-250"]:
    rpath = f"craft_output/results_{ck_name.replace('-','')}.json"
    if not os.path.exists(rpath):
        print(f"{ck_name:<18} {'NOT EVALUATED':>50}")
        continue
    with open(rpath) as f:
        res = json.load(f)
    
    gsm = res['gsm8k']['summary']['accuracy']
    strat = res['strategyqa']['summary']['accuracy']
    d_gsm = gsm - baseline_gsm
    d_strat = strat - baseline_strat
    avg_d = (d_gsm + d_strat) / 2
    all_results[ck_name] = res
    
    # Check Samsung criteria
    passes_gsm = gsm >= 0.50 and d_gsm >= 0.05
    passes_strat = strat >= 0.65 and d_strat >= 0.05
    passes_count = sum([passes_gsm, passes_strat])  # Need ≥2 of 3 benchmarks
    samsung_ok = passes_count >= 1  # At least 1 passes (realistically need 2)
    samsung_label = "✅ PASS" if passes_count >= 2 else ("⚠️ 1/2" if passes_count == 1 else "❌ FAIL")
    
    print(f"{ck_name:<18} {gsm:>7.1%} {d_gsm:>+7.1%} {strat:>8.1%} {d_strat:>+8.1%} {avg_d:>+6.1%} {samsung_label:>9}")
    
    if avg_d > best_avg_delta:
        best_avg_delta = avg_d
        best_checkpoint = ck_name

print("-" * 70)
print(f"\n  🏆 CHAMPION: {best_checkpoint} (avg improvement: {best_avg_delta:+.1%})")
print(f"{'='*70}\n")

# Save champion selection
champion_info = {
    "champion": best_checkpoint,
    "avg_improvement_percent": round(best_avg_delta * 100, 2),
    "baseline_gsm8k": round(baseline_gsm, 4),
    "baseline_strategyqa": round(baseline_strat, 4),
    "samsung_min_improvement_pct": 5.0,
    "passed": best_avg_delta >= 0.05,
}
with open("craft_output/champion_selection.json", "w") as f:
    json.dump(champion_info, f, indent=2)

CHAMPION_CHECKPOINT = best_checkpoint
CHAMPION_PATH = f"checkpoints/rl/{best_checkpoint}"
print(f"Champion checkpoint path: {CHAMPION_PATH}")
print(f"Champion info saved to craft_output/champion_selection.json")

---## Step 6: Merge + GGUF Convert + Quantize (CHAMPION ONLY)Merges the champion's LoRA weights, converts to GGUF, and quantizes to Q4_K_M for deployment.

In [ ]:
# Only quantize the champion — don't waste time on others
print(f"Merging and quantizing: {CHAMPION_CHECKPOINT}")

!python -m src.phase3_deploy.quantizer \
    --base-model microsoft/Phi-3-mini-4k-instruct \
    --lora-adapter {CHAMPION_PATH} \
    --merged-output craft_output/merged \
    --gguf-f16 craft_output/craft_phi3_f16.gguf \
    --gguf-quantized craft_output/craft_phi3_Q4_K_M.gguf

import os
assert os.path.exists("craft_output/craft_phi3_Q4_K_M.gguf"), \
    "GGUF not found! Check quantizer output above."
size_mb = os.path.getsize("craft_output/craft_phi3_Q4_K_M.gguf") / (1024*1024)
print(f"\n✅ Quantization complete! Size: {size_mb:.0f} MB")

---## Step 7: GGUF Deployment VerificationRuns a quick 30-sample sanity check on the GGUF to make sure quantization didn't destroy quality.The **official** Samsung score is from Step 4 (HF format).

In [ ]:
# This verifies the GGUF doesn't lose too much quality vs HF version
# Use 30 samples — just a sanity check, not the official score
# The OFFICIAL score for Samsung is from Cell 4 (HF format)

!python -m src.phase3_deploy.evaluator \
    --model-type gguf \
    --model-path craft_output/craft_phi3_Q4_K_M.gguf \
    --dataset all \
    --samples 30 \
    --label "CRAFT-GGUF-Q4" \
    --gpu \
    --output-json craft_output/gguf_verification_results.json

import json
with open("craft_output/gguf_verification_results.json") as f:
    gguf_res = json.load(f)

with open(f"craft_output/results_{CHAMPION_CHECKPOINT.replace('-','')}.json") as f:
    hf_res = json.load(f)

print("\n📊 GGUF Quality Verification (30 samples):")
for ds in ["gsm8k", "strategyqa"]:
    hf_acc = hf_res[ds]['summary']['accuracy']
    gguf_acc = gguf_res[ds]['summary']['accuracy']
    delta = gguf_acc - hf_acc
    print(f"  {ds}: HF={hf_acc:.1%} → GGUF={gguf_acc:.1%} (delta={delta:+.1%})")
    if abs(delta) > 0.05:
        print(f"  ⚠️  GGUF degraded by more than 5%! Consider Q5_K_M quantization instead.")
    else:
        print(f"  ✅ GGUF quality retained.")

---## Step 8: Generate Full Benchmark ReportCreates a professional Markdown report from **real** evaluation numbers. Nothing hardcoded.

In [ ]:
import json

with open("craft_output/champion_selection.json") as f:
    champ = json.load(f)

champion_label = champ["champion"]
champion_json = f"craft_output/results_{champion_label.replace('-','')}.json"

!python -m src.phase3_deploy.report_generator \
    --baseline-json craft_output/baseline_results.json \
    --champion-json {champion_json} \
    --champion-label "{champion_label}" \
    --ckpt150-json craft_output/results_checkpoint150.json \
    --ckpt200-json craft_output/results_checkpoint200.json \
    --ckpt250-json craft_output/results_checkpoint250.json \
    --report-md craft_output/benchmark_report.md

# Also copy to docs/ for Samsung submission
!cp craft_output/benchmark_report.md docs/benchmark_results.md
print("✅ Report generated and copied to docs/benchmark_results.md")

---## Step 9: Upload to HuggingFace (Gated on Samsung's Bar)This cell will **only upload** if the champion checkpoint passes Samsung's minimum improvement requirements.

In [ ]:
import json
from kaggle_secrets import UserSecretsClient

with open("craft_output/champion_selection.json") as f:
    champ = json.load(f)

# Samsung requires +5% on at least 2 benchmarks
# Use the champion's full results for the precise gate
champion_json_path = f"craft_output/results_{champ['champion'].replace('-','')}.json"
with open(champion_json_path) as f:
    champion_res = json.load(f)
with open("craft_output/baseline_results.json") as f:
    baseline_res = json.load(f)

gsm_delta = champion_res['gsm8k']['summary']['accuracy'] - baseline_res['gsm8k']['summary']['accuracy']
strat_delta = champion_res['strategyqa']['summary']['accuracy'] - baseline_res['strategyqa']['summary']['accuracy']
gsm_abs = champion_res['gsm8k']['summary']['accuracy']
strat_abs = champion_res['strategyqa']['summary']['accuracy']

passes_gsm = gsm_abs >= 0.50 and gsm_delta >= 0.05
passes_strat = strat_abs >= 0.65 and strat_delta >= 0.05
passes_count = sum([passes_gsm, passes_strat])

print(f"\n📊 Samsung Gate Check:")
print(f"  GSM8K:      {gsm_abs:.1%} (need ≥50%) | Δ={gsm_delta:+.1%} (need ≥+5%) → {'✅' if passes_gsm else '❌'}")
print(f"  StrategyQA: {strat_abs:.1%} (need ≥65%) | Δ={strat_delta:+.1%} (need ≥+5%) → {'✅' if passes_strat else '❌'}")
print(f"  Benchmarks passing: {passes_count}/2 (need ≥2)")

if passes_count < 2:
    print(f"\n🛑 UPLOAD BLOCKED. Model does not meet Samsung's minimum targets.")
    print(f"   Consider using an earlier checkpoint or retraining.")
else:
    print(f"\n✅ Model passes Samsung's bar. Proceeding with upload...")
    
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    if not hf_token:
        print("⚠️  HF_TOKEN not in Kaggle Secrets. Cannot upload.")
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        repo_id = "Aurvion/CRAFT-Phi3-Mini"
        api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
        
        # Upload GGUF model
        api.upload_file(
            path_or_fileobj="craft_output/craft_phi3_Q4_K_M.gguf",
            path_in_repo="craft_phi3_Q4_K_M.gguf",
            repo_id=repo_id
        )
        print(f"  ✅ GGUF model uploaded")
        
        # Upload benchmark report
        api.upload_file(
            path_or_fileobj="craft_output/benchmark_report.md",
            path_in_repo="benchmark_report.md",
            repo_id=repo_id
        )
        print(f"  ✅ Benchmark report uploaded")
        
        # Upload champion selection JSON (raw evidence)
        api.upload_file(
            path_or_fileobj="craft_output/champion_selection.json",
            path_in_repo="champion_selection.json",
            repo_id=repo_id
        )
        print(f"  ✅ Champion selection evidence uploaded")
        
        # Upload all checkpoint result JSONs
        for ck in ["checkpoint-150", "checkpoint-200", "checkpoint-250"]:
            rp = f"craft_output/results_{ck.replace('-','')}.json"
            if os.path.exists(rp):
                api.upload_file(
                    path_or_fileobj=rp,
                    path_in_repo=f"eval_results/{ck}.json",
                    repo_id=repo_id
                )
        print(f"  ✅ All evaluation JSONs uploaded")
        
        print(f"\n🚀 Upload complete: https://huggingface.co/{repo_id}")
        print(f"   You still need to add a model card (README.md) to HuggingFace.")
        print(f"   See Part 5 of CRAFT_Phase3_Fix.md for the model card template.")

---## Step 10: Package Delivery BundleCreates the final tar.gz bundle for Samsung submission.

In [ ]:
!python -m src.phase3_deploy.packager \
    --gguf craft_output/craft_phi3_Q4_K_M.gguf \
    --dir craft_delivery \
    --archive craft_delivery_bundle.tar.gz

import os
size_mb = os.path.getsize("craft_delivery_bundle.tar.gz") / (1024*1024)
print(f"✅ Delivery bundle: craft_delivery_bundle.tar.gz ({size_mb:.0f} MB)")